# FIT4112 · LAB 4 · Notebook 02
## Tạo lag, chia theo thời gian và baseline

Tại thời điểm t, mục tiêu là **PM2.5(t)**. Các đặc trưng gồm PM2.5(t−1), PM2.5(t−3), PM2.5(t−24) và giờ trong ngày (biết trước). Đây là bài toán rolling **one-hour-ahead**: quan sát mới được cập nhật từng giờ.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src" / "lab4.py").exists()), None)
assert ROOT is not None, "Mở notebook trong thư mục repo fit4112_lab_4."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project:", ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.lab4 import make_model_frame, time_split, baseline_predict, error_metrics

path = ROOT / "data/processed/hourly_pm25.csv"
assert path.exists(), "Hãy chạy Notebook 01 trước."
hourly = pd.read_csv(path, parse_dates=["datetime"])
frame = make_model_frame(hourly)
print("Bảng học máy:", frame.shape)
display(frame.head())
assert "pm25" not in frame.columns, "Không đưa giá trị đo cùng giờ vào features."

### 1. Chia train/test theo trục thời gian

**Không shuffle** và không sử dụng train_test_split ngẫu nhiên. Train trước, test sau. Đừng chỉnh sửa mô hình dựa trên sai số test nhiều lần.

In [ ]:
train, test = time_split(frame, test_fraction=0.2)
print("TRAIN:", train["datetime"].min(), "->", train["datetime"].max(), "|", len(train), "mẫu")
print("TEST :", test["datetime"].min(), "->", test["datetime"].max(), "|", len(test), "mẫu")
assert train["datetime"].max() < test["datetime"].min()

out = ROOT / "data/processed"
frame.to_csv(out / "model_frame.csv", index=False)

### 2. Persistence baseline

Dự đoán tại giờ t bằng **giá trị gần nhất đã biết ở t−1**. Đây là baseline mạnh với cảm biến có tính liên tục. Nếu mô hình học máy không tốt hơn baseline, hãy giải thích, **không sửa số liệu**.

In [ ]:
baseline_pred = baseline_predict(test)
baseline_scores = error_metrics(test["y"], baseline_pred)
display(pd.DataFrame([{"model": "Persistence", **baseline_scores}]).round(3))
pd.DataFrame({
    "datetime": test["datetime"], "actual": test["y"],
    "baseline": baseline_pred
}).to_csv(out / "baseline_predictions.csv", index=False)

sample = test.iloc[:min(100, len(test))].copy()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sample["datetime"], sample["y"], label="Actual")
ax.plot(sample["datetime"], baseline_pred[:len(sample)], label="Baseline", alpha=.85)
ax.set(title="Persistence baseline – first test hours", ylabel="PM2.5")
ax.legend()
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

### Câu hỏi
1. Thử giải thích một hàng: lag_1 khác y thế nào? Vì sao không đưa y vào features?
2. Vì sao cần baseline trước khi huấn luyện mô hình phức tạp?
3. Test nằm sau train có ý nghĩa gì? Liệu kết quả này có phải dự báo **24 giờ liên tục không có quan sát mới** không?